# 06 · item2vec và vector phiên (ticket #15)

**Input** (Add Input): Kaggle Dataset `titeohihi/mixnext-splited` (output của notebook 03).

**Output** (`/kaggle/working/out/`):
- `candidates/item2vec_{val,test}_run0.parquet`
- `item_embeddings/item2vec.npy`, `item_embeddings/item2vec_tracks.parquet`
- `item2vec_params.json`: tham số đã chọn và NDCG@10 trên val của cả lưới (lần chạy 1, 2 ở #26 dùng lại file này qua `--params`)

Word2Vec skip-gram (gensim) trên chuỗi bài của mỗi playlist train; negative 5, mọi bài train đều có vector. Vector phiên và điểm gu giống ALS (#14, `src/models/session.py`).

Lưới chọn trên val (vòng 2): window 3 × vector_size 128 × epochs {5, 20} × ns_exponent {0,75; −0,5} × normalize {không, có} × decay {0,5; 0,7; 0,9; 1,0} (32 cấu hình, 4 lần học Word2Vec; normalize và decay chỉ đổi cách chấm điểm). Chọn theo NDCG@10. Vòng 1 ở mục Kết quả.

`normalize`: chuẩn hóa mọi vector bài về độ dài 1 trước khi tính vector phiên và điểm gu, tức điểm gu là tổng có trọng số của cosine thay vì tích vô hướng thô (lệch nhỏ so với spec §6, ghi trong báo cáo). File `item2vec.npy` khi đó chứa vector đã chuẩn hóa, nên app vẫn chỉ cần tích vô hướng.

Học nhiều luồng nên cùng `random_seed` không ra cùng vector từng bit; khác biệt giữa các lần chạy đo ở #26.

gensim có sẵn trên Kaggle (#2). Bật **Internet** để clone repo. Chỉ cần CPU. Ước tính 35–45 phút.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #15 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "b"
!rm -rf /kaggle/working/MusicRecSys && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys /kaggle/working/MusicRecSys
import sys; sys.path.insert(0, "/kaggle/working/MusicRecSys")

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("eval_targets.parquet"))[0].parent
OUT = Path("/kaggle/working/out")
print("input:", DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
import time
from src.models.item2vec import main

t0 = time.time()
report = main(["--data", str(DATA), "--out", str(OUT)])   # chọn tham số trên val, xuất file, có validate_dataset bên trong
print(f"{time.time() - t0:.0f} giây")
for p in sorted(OUT.rglob("*.*")):
    print(p.relative_to(OUT), f"{p.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Lưới trên val, xếp theo NDCG@10, so với baseline (#9) và ALS (#14), cùng 9.212 phiên val
import pandas as pd

REFERENCE_NDCG = {"popularity": 0.0040, "itemknn": 0.1297, "als": 0.0967}
grid = pd.DataFrame(report["grid"]).sort_values("ndcg", ascending=False)
print(grid.round(4).to_string(index=False))
ndcg = report["val"]["ndcg"]
print(f"\nChọn: {report['params']} | NDCG@10 val {ndcg:.4f}")
for name, ref in REFERENCE_NDCG.items():
    print(f"  so với {name} {ref}: {ndcg / ref - 1:+.1%}")

## Đăng output

1. **Save Version → Save & Run All (Commit)** để lưu output.
2. Tab Output của version vừa chạy → **New Dataset** tên `mixnext-item2vec`; chỉ chọn thư mục `out`, bỏ `MusicRecSys`.
   - Không thêm được vào `mixnext-embeddings`: Dataset tạo từ output notebook chỉ nhận version mới từ chính notebook đó (notebook 05).
   - Notebook sau Add Input bộ này và tìm file bằng `Path("/kaggle/input").rglob(<tên file>)`.

## Kết quả trên val

### Vòng 1 (Kaggle, CPU, 26/11/2026, 1.754 giây): window {3, 10} × vector_size {128, 256} × decay, tham số mặc định của gensim (epochs 5, ns_exponent 0,75), tích vô hướng thô

| window | vector_size | decay | HitRate@10 | NDCG@10 | MRR |
|---:|---:|---:|---:|---:|---:|
| 3 | 128 | 0,5 | 0,0556 | **0,0282** | 0,0248 |
| 3 | 256 | 0,5 | 0,0526 | 0,0272 | 0,0244 |
| 10 | 256 | 0,5 | 0,0532 | 0,0266 | 0,0233 |
| 10 | 128 | 0,5 | 0,0521 | 0,0265 | 0,0237 |
| 3 | 128 | 0,7 | 0,0506 | 0,0256 | 0,0229 |
| 3 | 256 | 0,7 | 0,0481 | 0,0240 | 0,0216 |
| 10 | 128 | 0,7 | 0,0485 | 0,0236 | 0,0206 |
| 10 | 256 | 0,7 | 0,0479 | 0,0233 | 0,0205 |
| 3 | 128 | 0,9 | 0,0461 | 0,0228 | 0,0205 |
| 3 | 128 | 1,0 | 0,0448 | 0,0221 | 0,0199 |
| 3 | 256 | 0,9 | 0,0448 | 0,0219 | 0,0197 |
| 10 | 128 | 0,9 | 0,0433 | 0,0212 | 0,0189 |
| 3 | 256 | 1,0 | 0,0430 | 0,0207 | 0,0186 |
| 10 | 256 | 0,9 | 0,0422 | 0,0207 | 0,0186 |
| 10 | 128 | 1,0 | 0,0417 | 0,0206 | 0,0184 |
| 10 | 256 | 1,0 | 0,0404 | 0,0199 | 0,0179 |

Tốt nhất (3; 128; 0,5): NDCG@10 val 0,0282, hơn Popularity (0,0040) 7 lần nhưng kém ALS (0,0967) 71% và item-kNN liền kề (0,1297) 78%.

- window và vector_size gần như không ảnh hưởng (chênh khoảng 0,001), nên mở rộng hai tham số này không đáng; vòng 2 giữ (3, 128).
- decay 0,5 tốt nhất ở mọi cặp, giống ALS: vector không bị dồn về một hướng chung như trên dữ liệu giả.
- Nghi vấn: tích vô hướng thô bị độ dài vector (đi theo tần suất bài) chi phối, và tham số mặc định của gensim đặt cho văn bản. Vòng 2 thử chuẩn hóa vector, epochs 20 và ns_exponent âm (Caselles-Dupré và cộng sự 2018, "Word2vec applied to recommendation: hyperparameters matter").

### Vòng 2 (Kaggle, CPU, 26/11/2026, 1.999 giây): window 3 × vector_size 128 × epochs {5, 20} × ns_exponent {0,75; −0,5} × chuẩn hóa × decay

| epochs | ns_exponent | chuẩn hóa | decay | HitRate@10 | NDCG@10 | MRR |
|---:|---:|:---:|---:|---:|---:|---:|
| 20 | −0,5 | không | 0,5 | 0,1242 | **0,0672** | 0,0567 |
| 20 | −0,5 | không | 0,7 | 0,1186 | 0,0646 | 0,0554 |
| 20 | −0,5 | không | 0,9 | 0,1132 | 0,0603 | 0,0514 |
| 20 | −0,5 | không | 1,0 | 0,1093 | 0,0585 | 0,0500 |
| 20 | −0,5 | có | 0,5 | 0,0988 | 0,0577 | 0,0492 |
| 20 | 0,75 | không | 0,5 | 0,0954 | 0,0549 | 0,0477 |
| 20 | −0,5 | có | 0,7 | 0,0858 | 0,0492 | 0,0418 |
| 20 | 0,75 | không | 0,7 | 0,0875 | 0,0485 | 0,0418 |
| 5 | 0,75 | có | 0,5 | 0,0859 | 0,0482 | 0,0409 |
| 20 | −0,5 | có | 0,9 | 0,0761 | 0,0437 | 0,0375 |
| 20 | 0,75 | có | 0,5 | 0,0723 | 0,0428 | 0,0371 |
| 20 | −0,5 | có | 1,0 | 0,0736 | 0,0420 | 0,0357 |
| 20 | 0,75 | không | 0,9 | 0,0764 | 0,0414 | 0,0359 |
| 5 | 0,75 | có | 0,7 | 0,0729 | 0,0404 | 0,0345 |
| 20 | 0,75 | không | 1,0 | 0,0728 | 0,0390 | 0,0337 |
| 5 | −0,5 | có | 0,5 | 0,0637 | 0,0352 | 0,0300 |
| 20 | 0,75 | có | 0,7 | 0,0596 | 0,0344 | 0,0298 |
| 5 | 0,75 | có | 0,9 | 0,0625 | 0,0338 | 0,0288 |
| 5 | 0,75 | có | 1,0 | 0,0571 | 0,0310 | 0,0268 |
| 5 | −0,5 | có | 0,7 | 0,0557 | 0,0305 | 0,0262 |
| 20 | 0,75 | có | 0,9 | 0,0523 | 0,0296 | 0,0254 |
| 20 | 0,75 | có | 1,0 | 0,0493 | 0,0280 | 0,0242 |
| 5 | 0,75 | không | 0,5 | 0,0538 | 0,0275 | 0,0246 |
| 5 | −0,5 | có | 0,9 | 0,0493 | 0,0266 | 0,0230 |
| 5 | 0,75 | không | 0,7 | 0,0508 | 0,0253 | 0,0224 |
| 5 | −0,5 | có | 1,0 | 0,0446 | 0,0243 | 0,0216 |
| 5 | 0,75 | không | 0,9 | 0,0472 | 0,0228 | 0,0201 |
| 5 | 0,75 | không | 1,0 | 0,0465 | 0,0222 | 0,0195 |
| 5 | −0,5 | không | 0,5 | 0,0306 | 0,0150 | 0,0134 |
| 5 | −0,5 | không | 0,7 | 0,0296 | 0,0145 | 0,0131 |
| 5 | −0,5 | không | 1,0 | 0,0294 | 0,0143 | 0,0127 |
| 5 | −0,5 | không | 0,9 | 0,0292 | 0,0143 | 0,0127 |

**Chốt: window 3, vector_size 128, epochs 20, ns_exponent −0,5, không chuẩn hóa, decay 0,5** (`item2vec_params.json`). NDCG@10 val 0,0672: gấp 2,4 lần vòng 1 (0,0282), hơn Popularity (0,0040) 17 lần, nhưng kém ALS (0,0967) 30,5% và item-kNN liền kề (0,1297) 48%.

- epochs là yếu tố chính: tham số mặc định của gensim (5 epoch) học chưa đủ trên dữ liệu này.
- ns_exponent −0,5 (lấy mẫu âm nghiêng về bài ít phổ biến) chỉ có lợi khi học đủ: với 20 epoch hơn 0,75 (0,0672 so với 0,0549), với 5 epoch thì kém hơn. Khớp với Caselles-Dupré và cộng sự (2018).
- Chuẩn hóa vector chỉ bù cho việc học thiếu: với 5 epoch và ns_exponent 0,75 tăng từ 0,0275 lên 0,0482, nhưng ở cấu hình tốt nhất lại giảm (0,0672 → 0,0577). Vì vậy không chuẩn hóa, điểm gu giữ đúng tích vô hướng như spec §6, và không cần thử chuẩn hóa cho ALS.
- decay 0,5 tốt nhất ở mọi cấu hình, giống ALS và vòng 1.
- Giới hạn: epochs 20 vẫn ở mép lưới; thêm epoch có thể còn tăng nhưng dừng ở đây để dành thời gian cho SASRec (#16). ALS vẫn là mô hình embedding tốt hơn, nên app (#20, #21) dùng embedding ALS.
- `item2vec.npy` 128 chiều khoảng 67 MB.